In [1]:
import os, re
import numpy as np
import scanpy as sc
from os.path import join
import pandas as pd

import sys
import scipy.io as sio
import scipy.sparse as sps
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from spamosaic.framework import SpaMosaic
import spamosaic.utils as utls
from spamosaic.preprocessing import RNA_preprocess, ADT_preprocess, Epigenome_preprocess, harmony

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8' 

### Simulation

In [2]:
ds = 'Simulation'

data_dir = '../../../data/processed/Simulations'
fs = list(sorted(os.listdir(data_dir)))
fs

['rna-0.1-12-adt-2.1',
 'rna-0.2-10-adt-2.2',
 'rna-0.3-7-adt-2.3',
 'rna-0.4-5-adt-2.4',
 'rna-0.5-4-adt-2.5']

In [3]:
for fn in fs:
    for seed in [1,2,3]:
        _dir = join(data_dir, fn, str(seed))
        ad_mult_rna = sc.read_h5ad(join(_dir, 'ad_rna1.h5ad'))
        ad_mult_adt = sc.read_h5ad(join(_dir, 'ad_adt1.h5ad'))
        ad_test_rna = sc.read_h5ad(join(_dir, 'ad_rna2.h5ad'))
        ad_test_adt = sc.read_h5ad(join(_dir, 'ad_adt2.h5ad'))
        
        ad_mult_rna.X = ad_mult_rna.layers['counts'].copy()
        ad_mult_adt.X = ad_mult_adt.layers['counts'].copy()
        ad_test_rna.X = ad_test_rna.layers['counts'].copy()
        ad_test_adt.X = ad_test_adt.layers['counts'].copy()

        # 添加 src 标签
        ad_mult_rna.obs['src'] = 's1'
        ad_mult_adt.obs['src'] = 's1'
        ad_test_rna.obs['src'] = 's2'
        ad_test_adt.obs['src'] = 's3'

        # 给 test 的 obs_names 加前缀
        ad_test_rna.obs_names = [f"test-mod1-{i}" for i in range(ad_test_rna.n_obs)]
        ad_test_adt.obs_names = [f"test-mod2-{i}" for i in range(ad_test_adt.n_obs)]

        # concat
        ad_rna = sc.concat([ad_mult_rna, ad_test_rna])
        ad_adt = sc.concat([ad_mult_adt, ad_test_adt])

        sc.pp.normalize_total(ad_rna, target_sum=1e4)
        sc.pp.log1p(ad_rna)
        sc.pp.scale(ad_rna)
        sc.pp.pca(ad_rna, n_comps=50)
        ad_rna.obsm['X_pca_har'] = harmony(
            ad_rna.obsm['X_pca'],
            ad_rna.obs['src'].to_list(),
            use_gpu=True
        )
        utls.split_adata_ob([ad_mult_rna, ad_test_rna], ad_rna, ob='obsm', key='X_pca_har')

        sc.pp.normalize_total(ad_adt, target_sum=1e4)
        sc.pp.log1p(ad_adt)
        sc.pp.scale(ad_adt)
        sc.pp.pca(ad_adt, n_comps=50)
        ad_adt.obsm['X_pca_har'] = harmony(
            ad_adt.obsm['X_pca'],
            ad_adt.obs['src'].to_list(),
            use_gpu=True
        )
        utls.split_adata_ob([ad_mult_adt, ad_test_adt], ad_adt, ob='obsm', key='X_pca_har')
        
        input_dict = { 
            'rna':  [ad_mult_rna,  ad_test_rna,  None],
            'adt':  [ad_mult_adt, None       ,  ad_test_adt],
        }
        input_key = 'X_pca_har'
        
        def stack(xl, key):
            xs, ns = [], []
            for adx in xl:
                if adx is not None:
                    xs.append(adx.obsm[key])
                    ns.append(adx.obs_names)
            df = pd.DataFrame(np.vstack(xs), index=np.hstack(ns))
            return df
        
        for m1, m2 in zip(['rna', 'adt'], ['RNA', 'Protein']):
            fig_dir = f'../../../results/embeddings/Leiden-{m2}/Simulations/{fn}/{seed}'
            os.makedirs(fig_dir, exist_ok=True)
            df = stack(input_dict[m1], input_key)
            df.to_csv(join(fig_dir, 'df_emb.csv'))

Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
	Completed 4 / 10 iteration(s).
	Completed 5 / 10 iteration(s).
	Completed 6 / 10 iteration(s).
Reach convergence after 6 iteration(s).
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
	Completed 4 / 10 iteration(s).
	Completed 5 / 10 iteration(s).
	Completed 6 / 10 iteration(s).
Reach convergence after 6 iteration(s).
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
	Completed 4 / 10 iteration(s).
Reach convergence after 4 iteration(s).
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
	Completed 4 / 10 iteration(s).
	Completed 5 / 10 iteration(s).
	Completed 6 / 10 iteration(s).
